In [1]:
import time
import pdal
import pystac_client
import planetary_computer
from shapely.geometry import box, mapping
import geopandas as gpd

import pyproj
from shapely.geometry import Polygon
import rasterio
import numpy as np


In [2]:
nyc_bbox = box(-74.01, 40.75, -73.86, 40.88)
gdf_nyc = gpd.GeoDataFrame(geometry=[nyc_bbox], crs="EPSG:4326")
gdf_nyc.explore()

# Convert the bounding box to a GeoJSON dict for the STAC query
nyc_geojson = mapping(nyc_bbox)

# Open the STAC API client with sign-in
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

def get_items_with_retry(search_obj, max_attempts=3, wait_seconds=10):
    attempts = 0
    while attempts < max_attempts:
        try:
            items = list(search_obj.items())
            return items
        except Exception as e:
            print("Error occurred:", e)
            attempts += 1
            print(f"Retrying in {wait_seconds} seconds... (Attempt {attempts} of {max_attempts})")
            time.sleep(wait_seconds)
    raise Exception("Max retry attempts reached; please try again later or contact planetarycomputer@microsoft.com.")

# Create a STAC search query using the NYC GeoJSON polygon
search = catalog.search(
    collections=["3dep-lidar-copc"],
    intersects=nyc_geojson
)

# Use the retry function to get the items
items = get_items_with_retry(search)
print("Number of COPC tiles found for NYC:", len(items))

# Define output variables for PDAL
OUTPUT_RESOLUTION = 2.0
READ_RESOLUTION = 2.0

Error occurred: The request exceeded the maximum allowed time, please try again. If the issue persists, please contact planetarycomputer@microsoft.com.


Retrying in 10 seconds... (Attempt 1 of 3)
Error occurred: The request exceeded the maximum allowed time, please try again. If the issue persists, please contact planetarycomputer@microsoft.com.


Retrying in 10 seconds... (Attempt 2 of 3)
Error occurred: <!DOCTYPE html PUBLIC '-//W3C//DTD XHTML 1.0 Transitional//EN' 'http://www.w3.org/TR/xhtml1/DTD/xhtml1-transitional.dtd'>
<html xmlns='http://www.w3.org/1999/xhtml'>

<head>
    <meta content='text/html; charset=utf-8' http-equiv='content-type' />
    <style type='text/css'>
        body {
            font-family: Arial;
            margin-left: 40px;
        }

        img {
            border: 0 none;
        }

        #content {
            margin-left: auto;
            margin-right: auto
        }

        #message h2 {
            font-size: 20px;
            font-weight: norm

Exception: Max retry attempts reached; please try again later or contact planetarycomputer@microsoft.com.

In [ ]:
transformer = pyproj.Transformer.from_crs("EPSG:4326", "EPSG:32618", always_xy=True)
nyc_bbox_utm_coords = [transformer.transform(x, y) for x, y in nyc_bbox.exterior.coords]
nyc_bbox_utm = Polygon(nyc_bbox_utm_coords)
polygon_str = nyc_bbox_utm.wkt + " / EPSG:32618"
print("Clipping polygon for PDAL:")
print(polygon_str)

# Build PDAL readers for each tile
readers = []
for tile in items:
    url = tile.assets["data"].href
    reader = pdal.Reader.copc(
        filename=url,
        requests=3,
        resolution=READ_RESOLUTION,
        polygon=polygon_str
    )
    readers.append(reader)

# Build the PDAL pipeline: merge the readers, compute HeightAboveGround (HAG) using hag_nn, and write a GeoTIFF.
pipeline = None
for reader in readers:
    if pipeline is None:
        pipeline = reader
    else:
        pipeline = pipeline | reader

merge = pdal.Filter.merge()
hag_filter = pdal.Filter.hag_nn()
writer = pdal.Writer.gdal(
    filename="nyc_hag.tif",
    resolution=OUTPUT_RESOLUTION,
    dimension="HeightAboveGround",
    data_type="float32",
    output_type="mean",
    nodata=-9999
)

pipeline = pipeline | merge | hag_filter | writer
p = pipeline.execute()
print("NYC HAG raster '../data/nyc_hag.tif' created.")

# (Optional) Apply a color ramp for visualization using gdaldem
colorramp = """-10,247,251,255
-0.001,228,239,249
0.172,209,226,243
0.675,186,214,235
1.704,154,200,224
3.806,115,178,216
6.603,82,157,204
13.593,53,133,191
35.150,29,108,177
68.496,8,81,156
347.239,8,48,107"""
with open("../data/nyc_hag_colors.txt", "w") as f:
    f.write(colorramp)

# Execute the gdaldem color-relief command (requires a shell environment)
get_ipython().system('gdaldem color-relief nyc_hag.tif nyc_hag_colors.txt nyc_hag.tif')

In [3]:
from PIL import Image
Image.open("../data/nyc_hag.tif").show()

In [4]:
with rasterio.open("../data/nyc_hag.tif") as src:
    hag_array = src.read(1, masked=True)
    print("HAG statistics:")
    print("Min:", np.min(hag_array))
    print("Max:", np.max(hag_array))
    print("Mean:", np.mean(hag_array))

HAG statistics:
Min: 8
Max: 247
Mean: 209.10022025402915
